In [ ]:
from pathlib import Path
import pandas as pd

DOSSIER_PET = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\collecte_PET_20261026_221901_669799"
)

pet = pd.read_csv(
    DOSSIER_PET / "series_PET_FDG_candidates.csv",
    dtype="string"
)

ids = (
    pet["image_id"]
    .str.strip()
    .str.replace(r"^[Ii]", "", regex=True)
)

if ids.isna().any() or not ids.str.fullmatch(r"\d+").all():
    raise ValueError("Identifiants PET absents ou invalides.")

if ids.duplicated().any():
    raise ValueError("Identifiants PET en double : à examiner.")

ids = sorted(ids.tolist(), key=int)

horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
LOTS = DOSSIER_PET / f"lots_PET_{horodatage}"
LOTS.mkdir(parents=True, exist_ok=False)

suivi = []

for debut in range(0, len(ids), 100):
    numero = debut // 100 + 1
    contenu = ids[debut:debut + 100]
    nom = f"PET_FDG_lot_{numero:03d}.txt"

    (LOTS / nom).write_text(",".join(contenu), encoding="utf-8")

    suivi.append({
        "lot": f"{numero:03d}",
        "series_attendues": len(contenu),
        "fichier_identifiants": nom,
        "statut": "a_telecharger"
    })

suivi = pd.DataFrame(suivi)
suivi.to_csv(LOTS / "suivi_telechargements_PET.csv", index=False)

display(suivi)
print("Dossier des listes :", LOTS)

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import re

PET = Path(r"E:\Projet_Alzheimer\data\PET")

COLLECTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\collecte_PET_20261026_221901_669799"
)

# Liste des images attendues
catalogue = pd.read_csv(
    COLLECTE / "series_PET_FDG_candidates.csv",
    dtype="string"
)

ids = (
    catalogue["image_id"].str.strip()
    .str.replace(r"^[Ii]", "", regex=True)
)

if ids.isna().any() or not ids.str.fullmatch(r"\d+").all():
    raise ValueError("Identifiants invalides dans le catalogue.")

attendus = set(ids)
fichiers_par_image = defaultdict(int)
lots_par_image = defaultdict(set)
bilan_lots = []
non_identifies = 0
vides = 0

# Examiner uniquement les dossiers d'images, pas les métadonnées
for numero in range(1, 12):
    lot = f"{numero:03d}"
    dossier = PET / lot
    images_lot = set()
    nombre_dcm = 0

    for fichier in dossier.rglob("*"):
        if not fichier.is_file() or fichier.suffix.lower() != ".dcm":
            continue

        nombre_dcm += 1

        if fichier.stat().st_size == 0:
            vides += 1

        correspondance = re.search(
            r"_I(\d+)(?=[_.]|$)",
            fichier.name,
            flags=re.IGNORECASE
        )

        if correspondance is None:
            for partie in reversed(fichier.relative_to(dossier).parts[:-1]):
                correspondance = re.fullmatch(
                    r"I(\d+)", partie, flags=re.IGNORECASE
                )
                if correspondance:
                    break

        if correspondance is None:
            non_identifies += 1
            continue

        image_id = correspondance.group(1)
        images_lot.add(image_id)
        fichiers_par_image[image_id] += 1
        lots_par_image[image_id].add(lot)

    bilan_lots.append({
        "lot": lot,
        "dossier_present": dossier.is_dir(),
        "fichiers_dcm": nombre_dcm,
        "images_identifiees": len(images_lot)
    })

presents = set(fichiers_par_image)
manquants = attendus - presents
inattendus = presents - attendus

display(pd.DataFrame(bilan_lots))

print("\nBILAN PET")
print("Images attendues :", len(attendus))
print("Images attendues retrouvées :", len(attendus & presents))
print("Images non retrouvées :", len(manquants))
print("Images inattendues :", len(inattendus))
print("Fichiers DICOM sans identifiant reconnu :", non_identifies)
print("Fichiers DICOM vides :", vides)
print("Images présentes dans plusieurs lots :",
      sum(len(lots) > 1 for lots in lots_par_image.values()))

# Conserver le résultat de la vérification
horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
SORTIE = COLLECTE / f"verification_PET_{horodatage}"
SORTIE.mkdir(parents=True, exist_ok=False)

pd.DataFrame(bilan_lots).to_csv(
    SORTIE / "bilan_lots.csv", index=False
)

rapport = pd.DataFrame([
    {
        "image_id": image_id,
        "attendue": image_id in attendus,
        "fichiers_dcm": fichiers_par_image.get(image_id, 0),
        "lots": ";".join(sorted(lots_par_image.get(image_id, set())))
    }
    for image_id in sorted(attendus | presents, key=int)
])

rapport.to_csv(SORTIE / "presence_images.csv", index=False)

catalogue.loc[ids.isin(manquants)].to_csv(
    SORTIE / "images_non_retrouvees.csv", index=False
)

print("\nRapports enregistrés :", SORTIE)

In [ ]:
from pathlib import Path
import pandas as pd

VERIFICATION = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\collecte_PET_20261026_221901_669799"
    r"\verification_PET_20261027_145915_313004"
)

manquantes = pd.read_csv(
    VERIFICATION / "images_non_retrouvees.csv",
    dtype="string"
)

ids = (
    manquantes["image_id"].str.strip()
    .str.replace(r"^[Ii]", "", regex=True)
)

if ids.isna().any() or not ids.str.fullmatch(r"\d+").all():
    raise ValueError("Identifiants absents ou invalides.")

ids = sorted(set(ids), key=int)
liste = ",".join(ids)

fichier = VERIFICATION / "PET_rattrapage_26.txt"
fichier.write_text(liste, encoding="utf-8")

print("Nombre d'images à rechercher :", len(ids))
print("Liste à copier dans Image ID :")
print(liste)
print("\nFichier enregistré :", fichier)

In [ ]:
from pathlib import Path
import pandas as pd

COLLECTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\collecte_PET_20261026_221901_669799"
)

VERIFICATION = COLLECTE / "verification_PET_20261027_145915_313004"

catalogue = pd.read_csv(
    COLLECTE / "series_PET_FDG_candidates.csv",
    dtype="string"
)

presence = pd.read_csv(
    VERIFICATION / "presence_images.csv",
    dtype="string"
)

# Harmoniser les identifiants
for table in (catalogue, presence):
    table["image_id"] = (
        table["image_id"].str.strip()
        .str.replace(r"^[Ii]", "", regex=True)
    )

nombre_fichiers = pd.to_numeric(
    presence["fichiers_dcm"], errors="raise"
)

ids_presents = set(
    presence.loc[nombre_fichiers > 0, "image_id"]
)

disponibles = catalogue.loc[
    catalogue["image_id"].isin(ids_presents)
].copy()

disponibles["date_pet"] = pd.to_datetime(
    disponibles["date_pet"], errors="raise"
).dt.normalize()

if disponibles["date_pet"].isna().any():
    raise ValueError("Une date PET est absente.")

participants = catalogue["PTID"].dropna().unique()

suivi = (
    disponibles.groupby("PTID")["date_pet"]
    .nunique()
    .reindex(participants, fill_value=0)
    .rename("nombre_dates_PET_disponibles")
)

print("Participants ciblés :", len(suivi))
print("Avec au moins 2 dates PET disponibles :", (suivi >= 2).sum())
print("Avec seulement 1 date PET disponible :", (suivi == 1).sum())
print("Sans image PET disponible :", (suivi == 0).sum())

horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
suivi.rename_axis("PTID").reset_index().to_csv(
    VERIFICATION / f"suivi_PET_disponible_{horodatage}.csv",
    index=False
)

In [ ]:
from pathlib import Path
import json
from datetime import datetime

COLLECTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\collecte_PET_20261026_221901_669799"
)

decision = {
    "date": datetime.now().isoformat(timespec="seconds"),
    "participants_cibles": 359,
    "images_PET_attendues": 1086,
    "images_PET_presentes": 1060,
    "images_PET_non_retrouvees": 26,
    "participants_avec_au_moins_2_dates_PET": 359,
    "decision": "Poursuivre sans rattrapage pour le moment",
    "controle_qualite_images": "A realiser"
}

destination = COLLECTE / (
    "decision_collecte_PET_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    + ".json"
)

destination.write_text(
    json.dumps(decision, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Décision enregistrée :", destination)

In [ ]:
from pathlib import Path
import pandas as pd
import pydicom
import warnings
import re

PET = Path(r"E:\Projet_Alzheimer\data\PET")
COLLECTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\collecte_PET_20261026_221901_669799"
)

# 1. Charger les informations attendues
catalogue = pd.read_csv(
    COLLECTE / "series_PET_FDG_candidates.csv",
    dtype="string"
)

catalogue["image_id"] = (
    catalogue["image_id"].str.strip()
    .str.replace(r"^[Ii]", "", regex=True)
)

# 2. Repérer un fichier DICOM par identifiant d'image
exemples = {}

for numero in range(1, 12):
    dossier = PET / f"{numero:03d}"

    for fichier in dossier.rglob("*"):
        if not fichier.is_file() or fichier.suffix.lower() != ".dcm":
            continue

        correspondance = re.search(
            r"_I(\d+)(?=[_.]|$)", fichier.name, flags=re.IGNORECASE
        )

        if correspondance is None:
            for partie in reversed(fichier.relative_to(dossier).parts[:-1]):
                correspondance = re.fullmatch(
                    r"I(\d+)", partie, flags=re.IGNORECASE
                )
                if correspondance:
                    break

        if correspondance:
            exemples.setdefault(correspondance.group(1), fichier)

print("Séries repérées :", len(exemples))

# 3. Lire les en-têtes sans charger les pixels
lignes = []

for numero, (image_id, fichier) in enumerate(
    sorted(exemples.items(), key=lambda item: int(item[0])), start=1
):
    ligne = {
        "image_id": image_id,
        "fichier_exemple": str(fichier),
        "patient_dicom": "",
        "date_dicom": "",
        "modalite": ""
    }

    with warnings.catch_warnings(record=True) as alertes:
        warnings.simplefilter("always")

        try:
            ds = pydicom.dcmread(fichier, stop_before_pixels=True)

            ligne.update({
                "statut": "OK",
                "patient_dicom": str(ds.get("PatientID", "")),
                "date_dicom": str(ds.get("StudyDate", "")),
                "modalite": str(ds.get("Modality", "")),
                "description": str(ds.get("SeriesDescription", "")),
                "series_uid": str(ds.get("SeriesInstanceUID", "")),
                "lignes_pixels": str(ds.get("Rows", "")),
                "colonnes_pixels": str(ds.get("Columns", "")),
                "espacement_pixels": str(ds.get("PixelSpacing", "")),
                "epaisseur_coupe": str(ds.get("SliceThickness", ""))
            })

        except Exception as erreur:
            ligne.update({
                "statut": "ERREUR",
                "erreur": f"{type(erreur).__name__}: {erreur}"
            })

        ligne["avertissements"] = " | ".join(
            sorted({str(alerte.message) for alerte in alertes})
        )

    lignes.append(ligne)

    if numero % 200 == 0:
        print(f"{numero} / {len(exemples)} en-têtes examinés")

if not lignes:
    raise ValueError("Aucune série repérée : vérifier le dossier PET.")

# 4. Comparer aux informations du catalogue
inventaire = pd.DataFrame(lignes)

controle = inventaire.merge(
    catalogue[["image_id", "PTID", "date_pet"]],
    on="image_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

controle["participant_entete"] = controle["patient_dicom"].str.extract(
    r"(\d{3}_S_\d{4})", expand=False
)

controle["date_entete"] = pd.to_datetime(
    controle["date_dicom"], format="%Y%m%d", errors="coerce"
)

controle["date_attendue"] = pd.to_datetime(
    controle["date_pet"], errors="coerce"
).dt.normalize()

controle["participant_concordant"] = (
    controle["PTID"].str.strip()
    .eq(controle["participant_entete"])
    .fillna(False)
)

controle["date_concordante"] = (
    controle["date_entete"].notna()
    & controle["date_attendue"].notna()
    & controle["date_entete"].eq(controle["date_attendue"])
)

# 5. Sauvegarder et afficher uniquement le bilan
horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
destination = COLLECTE / f"controle_entetes_PET_{horodatage}.csv"
controle.to_csv(destination, index=False)

print("\nBILAN PET")
print("Séries examinées :", len(controle))
print("En-têtes lus :", controle["statut"].eq("OK").sum())
print("Erreurs de lecture :", controle["statut"].eq("ERREUR").sum())
print("Séries présentes dans le catalogue :", controle["_merge"].eq("both").sum())
print("Participants concordants :", controle["participant_concordant"].sum())
print("Dates concordantes :", controle["date_concordante"].sum())
print("Séries avec avertissements :", controle["avertissements"].ne("").sum())

print("\nModalités :")
print(controle["modalite"].value_counts(dropna=False).to_string())

print("\nRapport enregistré :", destination)

In [ ]:
from pathlib import Path
import pandas as pd
import pydicom
import numpy as np
import warnings

COLLECTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
    r"\collecte_PET_20261026_221901_669799"
)

inventaire = pd.read_csv(
    COLLECTE / "controle_entetes_PET_20261027_151028_758875.csv",
    dtype="string"
)

horodatage = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
SORTIE = COLLECTE / f"lecture_pixels_PET_{horodatage}"
SORTIE.mkdir(parents=True, exist_ok=False)

bilans = []
anomalies = []

for numero, (_, serie) in enumerate(inventaire.iterrows(), start=1):
    dossier = Path(serie["fichier_exemple"]).parent
    fichiers = sorted(
        f for f in dossier.iterdir()
        if f.is_file() and f.suffix.lower() == ".dcm"
    )

    lus = 0
    erreurs = 0
    non_finis = 0
    formes = set()

    for fichier in fichiers:
        with warnings.catch_warnings(record=True) as alertes:
            warnings.simplefilter("always")

            try:
                ds = pydicom.dcmread(fichier)
                pixels = ds.pixel_array

                formes.add(str(pixels.shape))
                lus += 1

                if not np.isfinite(pixels).all():
                    non_finis += 1

                del pixels, ds

            except Exception as erreur:
                erreurs += 1
                anomalies.append({
                    "image_id": serie["image_id"],
                    "fichier": str(fichier),
                    "type": "erreur",
                    "message": f"{type(erreur).__name__}: {erreur}"
                })

            for alerte in alertes:
                anomalies.append({
                    "image_id": serie["image_id"],
                    "fichier": str(fichier),
                    "type": "avertissement",
                    "message": str(alerte.message)
                })

    bilans.append({
        "image_id": serie["image_id"],
        "fichiers_trouves": len(fichiers),
        "fichiers_decodes": lus,
        "erreurs": erreurs,
        "fichiers_valeurs_non_finies": non_finis,
        "formes_pixels": ";".join(sorted(formes))
    })

    # Sauvegarde progressive
    if numero % 50 == 0 or numero == len(inventaire):
        pd.DataFrame(bilans).to_csv(
            SORTIE / "bilan_pixels.csv", index=False
        )
        pd.DataFrame(
            anomalies,
            columns=["image_id", "fichier", "type", "message"]
        ).to_csv(SORTIE / "anomalies.csv", index=False)

        print(f"{numero} / {len(inventaire)} séries examinées")

bilan = pd.DataFrame(bilans)

print("\nBILAN DE LECTURE DES PIXELS")
print("Séries examinées :", len(bilan))
print("Fichiers trouvés :", bilan["fichiers_trouves"].sum())
print("Fichiers décodés :", bilan["fichiers_decodes"].sum())
print("Erreurs de décodage :", bilan["erreurs"].sum())
print("Dossiers sans DICOM :", bilan["fichiers_trouves"].eq(0).sum())
print("Fichiers avec valeurs non finies :",
      bilan["fichiers_valeurs_non_finies"].sum())

print("\nNombre de fichiers par série :")
print(bilan["fichiers_trouves"].value_counts().sort_index().to_string())

print("\nRapports enregistrés :", SORTIE)